Project14 - Tiny LLM Story Generator

**Purpose:** This notebook trains a compact GPT-2 style language model to generate short children’s stories using the


In [2]:
!pip install datasets transformers


In [4]:
from datasets import load_dataset


In [5]:
dataset = load_dataset("roneneldan/TinyStories")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…):   0%|          | 0.00/249M [00:00<?, ?B/s]

data/train-00001-of-00004-5852b56a2bd28f(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/train-00002-of-00004-a26307300439e9(…):   0%|          | 0.00/246M [00:00<?, ?B/s]

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

In [6]:
dataset = load_dataset("roneneldan/TinyStories")


In [7]:
# inspect data
print(dataset)
print(dataset["train"][0])


DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2119719
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 21990
    })
})
{'text': 'One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.'}


In [8]:
# Tokenization (Lightweight for Small Models)
from transformers import AutoTokenizer

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Make sure it has a pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Tokenize
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=128,   # small context window
        padding="max_length"
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Map:   0%|          | 0/21990 [00:00<?, ? examples/s]

In [9]:
# Prepare for Training
tokenized_dataset.set_format("torch")  # for PyTorch
# OR
tokenized_dataset.set_format("tensorflow")  # for TensorFlow


3. TinyStoriesStreamDataset Class

- Creates a **streaming PyTorch dataset** for TinyStories text.  
- Steps performed for each story:
  1. **Skip short samples:** Stories shorter than `min_length` are ignored.  
  2. **Clean text:**  
     - Removes extra spaces and unwanted characters.  
     - Replaces fancy quotes with standard quotes.  
  3. **Tokenize:** Converts text into token IDs using a GPT-2 tokenizer.  
  4. **Prepare training inputs:**  
     - `input_ids`: All tokens except the last one.  
     - `labels`: All tokens except the first one (for next-token prediction).  
     - `attention_mask`: Marks which tokens are real vs. padding.

In [10]:
from torch.utils.data import IterableDataset

class TinyStoriesStreamDataset(IterableDataset):
    def __init__(self, dataset_stream, tokenizer, block_size=512, min_length=30):
        self.dataset = dataset_stream
        self.tokenizer = tokenizer
        self.block_size = block_size
        self.min_length = min_length

    def __iter__(self):
        for sample in self.dataset:
            text = sample["text"].strip()
            if len(text) < self.min_length:
                continue

            text = re.sub(r'\s+', ' ', text)
            text = re.sub(r'[“”]', '"', text)
            text = re.sub(r"[‘’]", "'", text)
            text = re.sub(r'[^a-zA-Z0-9.,!?\'"\s]', '', text)

            tokenized = self.tokenizer(
                text,
                truncation=True,
                add_special_tokens=True,
                padding="max_length",
                max_length=self.block_size,
                return_tensors="pt"
            )

            input_ids = tokenized["input_ids"][0]
            attention_mask = tokenized["attention_mask"][0]

            yield {
                "input_ids": input_ids[:-1],
                "labels": input_ids[1:],
                "attention_mask": attention_mask[:-1]
            }

In [12]:
# Load Tokenizer, DataLoader, Model, and Optimizer Setup
from transformers import GPT2Tokenizer
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import GPT2Config, GPT2LMHeadModel
from tqdm.auto import tqdm
import torch


total_samples = 2119719
batch_size = 52
max_batches_per_epoch = total_samples // batch_size


tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

stream_dataset = TinyStoriesStreamDataset(dataset, tokenizer)
train_loader = DataLoader(stream_dataset, batch_size=batch_size)

config = GPT2Config(
    vocab_size=len(tokenizer),
    n_positions=512,
    n_ctx=512,
    n_embd=256,
    n_layer=4,
    n_head=4,
    pad_token_id=tokenizer.pad_token_id)


model = GPT2LMHeadModel(config)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)


if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs")
    model = torch.nn.DataParallel(model)

optimizer = AdamW(model.parameters(), lr=5e-5)

In [14]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Add pad token if missing
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=128,
        padding="max_length"
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])


Map:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Map:   0%|          | 0/21990 [00:00<?, ? examples/s]

In [15]:
def add_labels(example):
    example["labels"] = example["input_ids"].copy()
    return example

tokenized_dataset = tokenized_dataset.map(add_labels)


Map:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Map:   0%|          | 0/21990 [00:00<?, ? examples/s]

In [16]:
import torch
from torch.utils.data import DataLoader

train_dataset = tokenized_dataset["train"]
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)


In [18]:
tokenized_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])


In [19]:
from torch.utils.data import DataLoader

train_dataset = tokenized_dataset["train"]
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)


In [20]:
for i, batch in enumerate(tqdm(train_loader, total=max_batches_per_epoch)):
    input_ids = batch["input_ids"].to(device)
    labels = batch["labels"].to(device)
    attention_mask = batch["attention_mask"].to(device)


  0%|          | 0/40763 [00:00<?, ?it/s]

In [22]:
# Load & Tokenize Dataset
from datasets import load_dataset
from transformers import AutoTokenizer

# Load TinyStories
dataset = load_dataset("roneneldan/TinyStories")

# Load GPT-2 tokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Tokenize
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=128,
        padding="max_length"
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

# Add labels
def add_labels(example):
    example["labels"] = example["input_ids"].copy()
    return example

tokenized_dataset = tokenized_dataset.map(add_labels)


Map:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Map:   0%|          | 0/21990 [00:00<?, ? examples/s]

Map:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Map:   0%|          | 0/21990 [00:00<?, ? examples/s]

In [24]:
#Create DataLoader
import torch
from torch.utils.data import DataLoader
from transformers import DataCollatorForLanguageModeling

# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# Train loader
train_loader = DataLoader(
    tokenized_dataset["train"],
    batch_size=8,
    shuffle=True,
    collate_fn=data_collator
)

# Limit batches per epoch for faster testing
max_batches_per_epoch = len(train_loader)  # or set smaller like 200


In [28]:
from transformers import AutoModelForCausalLM
from torch.optim import AdamW


In [29]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained("gpt2").to(device)
optimizer = AdamW(model.parameters(), lr=5e-5)


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [31]:
# 1. Load dataset
from datasets import load_dataset
dataset = load_dataset("roneneldan/TinyStories")

# 2. Tokenizer
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 3. Tokenize + add labels
def tokenize_function(example):
    return tokenizer(example["text"], truncation=True, max_length=128, padding="max_length")
dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])
dataset = dataset.map(lambda e: {"labels": e["input_ids"]})

# 4. DataLoader
from torch.utils.data import DataLoader
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
train_loader = DataLoader(dataset["train"], batch_size=8, shuffle=True, collate_fn=data_collator)
max_batches_per_epoch = len(train_loader)  # or smaller, e.g. 500

# 5. Model + Optimizer
import torch
from transformers import AutoModelForCausalLM
from torch.optim import AdamW

device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForCausalLM.from_pretrained("gpt2").to(device)
optimizer = AdamW(model.parameters(), lr=5e-5)


Map:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Map:   0%|          | 0/21990 [00:00<?, ? examples/s]

Map:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Map:   0%|          | 0/21990 [00:00<?, ? examples/s]

### Generate Text from a Saved GPT-2 Checkpoint

In [43]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [44]:
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [45]:
!rm -rf /root/.config/Google/DriveFS
!rm -rf /root/.config/Google/drive


In [48]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [50]:
import os

base_dir = "/content/drive/MyDrive/TinyLLM/model"
if os.path.exists(base_dir):
    print("Checkpoints available:", os.listdir(base_dir))
else:
    print("Model folder not found! Check if you uploaded it to Drive.")


Model folder not found! Check if you uploaded it to Drive.


In [53]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel
import torch

# Load pre-trained GPT2 model & tokenizer
model_directory = "gpt2"  # This loads the official GPT2 from Hugging Face
tokenizer = GPT2Tokenizer.from_pretrained(model_directory)
model = GPT2LMHeadModel.from_pretrained(model_directory)
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def generate(input_text, max_len=30):
    tokenizer.pad_token = tokenizer.eos_token
    inputs = tokenizer(input_text, return_tensors='pt', padding=True).to(device)
    output = model.generate(
        input_ids=inputs['input_ids'],
        attention_mask=inputs['attention_mask'],
        max_length=max_len
    )
    return tokenizer.decode(output[0], skip_special_tokens=True)

# Test generation
print(generate("Once there was a little boy", 30))
print(generate("Once there was a little girl", 30))


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Once there was a little boy, he was a little boy. He was a little boy. He was a little boy. He was a little boy
Once there was a little girl, she was a little girl. She was a little girl. She was a little girl. She was a little girl


In [54]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel
import torch

# Load pre-trained GPT2 model & tokenizer
model_name = "gpt2"  # can also try "gpt2-medium" if GPU memory allows
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)
model.eval()

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Text generation function
def generate(input_text, max_len=50):
    tokenizer.pad_token = tokenizer.eos_token
    inputs = tokenizer(input_text, return_tensors='pt', padding=True).to(device)

    output = model.generate(
        input_ids=inputs['input_ids'],
        attention_mask=inputs['attention_mask'],
        max_length=max_len,
        do_sample=True,           # enables sampling
        top_k=50,                 # consider top 50 tokens at each step
        top_p=0.95,               # nucleus sampling
        temperature=0.8,          # randomness factor
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        no_repeat_ngram_size=2    # avoid repeating same phrases
    )

    generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
    return generated_text

# Test examples
print(generate("Once there was a little boy"))
print(generate("Once there was a little girl"))
print(generate("Once there was a cute little dog"))
print(generate("Once there was a handsome prince"))


Once there was a little boy, I heard a young man say that he heard that boy's name called. He came up to me and asked, "Who are you?" I said, 'I'm your boy.'" He said it was my son
Once there was a little girl with a face that looked like it was going to be a girl, we had to get her to put her on the floor, where the girl will get a nice, clean clothes. The girl doesn't have to do
Once there was a cute little dog in the corner, I just kept going back, 'Oh, no,' because I was so close to tears. I wasn't just saying no, but I had tears in my eyes. It's all just a
Once there was a handsome prince, and the prince was so young, so beautiful, that he had no reason to doubt that it was of all the kind. But that prince had a temper, he could not see and he refused to take any risks


In [55]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel
import torch

model_directory = "gpt2"  # use Hugging Face pre-trained GPT2

tokenizer = GPT2Tokenizer.from_pretrained(model_directory)
model = GPT2LMHeadModel.from_pretrained(model_directory)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def generate(input_text, max_len):
    tokenizer.pad_token = tokenizer.eos_token

    inputs = tokenizer(
        input_text,
        return_tensors='pt',
        padding=True,
        return_attention_mask=True
    ).to(device)

    output = model.generate(
        input_ids=inputs['input_ids'],
        attention_mask=inputs['attention_mask'],
        max_length=max_len,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.8,
        no_repeat_ngram_size=2,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
    return generated_text

print(generate("Once there was little boy",30))
print(generate("Once there was little girl",30))
print(generate("Once there was a cute",30))
print(generate("Once there was a cute little",30))
print(generate("Once there was a handsome",30))


Once there was little boy, she was a little girl, and she had two brothers. She had never met her father before."

Hil
Once there was little girl in the bathroom, she was getting carried away with a cold.

As a teenager, I got on a bus.
Once there was a cute lady wearing a bright green scarf to make up her hair, she could have been a hero.

Even though you can
Once there was a cute little girl, that was probably how she got her start as a cook.

Well, it's been a while since
Once there was a handsome man in the front seat of the sedan, the two men began to kiss.

"Well, I guess it would


###Inference with Pretrained TinyStories Model

In [56]:
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig

model = AutoModelForCausalLM.from_pretrained('roneneldan/TinyStories-3M')

tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neo-125M")

prompt = "Once upon a time there was"


def generate(input_text, max_len):

  tokenizer.pad_token = tokenizer.eos_token

  inputs = tokenizer(
      input_text,
      return_tensors='pt',
      padding=True,
      return_attention_mask=True
  )

  output = model.generate(
      input_ids=inputs['input_ids'],
      attention_mask=inputs['attention_mask'],
      max_length=max_len
  )

  generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
  return generated_text

  return output_text

print(generate("Once there was little boy",30))
print(generate("Once there was little girl",30))
print(generate("Once there was a cute",30))
print(generate("Once there was a cute little",30))
print(generate("Once there was a handsome",30))

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/66.7M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/66.7M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Once there was little girl who was three years old. She was very curious and wanted to explore the world.

One day, she decided to


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Once there was a cute little girl named Lily. She loved to play outside in the sunshine. One day, she saw a big, red ball in


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Once there was a cute little girl named Lily. She loved to play outside in the sunshine. One day, she saw a big, red ball in
Once there was a handsome boy named Tom. He was very brave and always wanted to help others. One day, Tom decided to go on an adventure


### Assignment - Task : Code-Focused Inference


1. **Load Model and Tokenizer:** Load a suitable pre-trained GPT-2 model and its corresponding tokenizer. You can use `transformers.AutoModelForCausalLM` and `transformers.AutoTokenizer`. A smaller model like `gpt2` or `gpt2-medium` might be sufficient.
2. **Implement a Filtering Mechanism:** Before generating a response, check if the input prompt is related to Python coding. You can use simple keyword matching (e.g., "Python", "code", "function", "class", "import") or a more sophisticated approach using a text classification model (optional).
3. **Generate Response:** If the prompt is deemed a Python coding question, generate a response using the loaded GPT-2 model.
4. **Handle Non-Coding Questions:** If the prompt is not related to Python coding, return a predefined message indicating that the model can only answer coding questions.
5. **Test:** Test your implementation with various prompts, including both Python coding questions and non-coding questions, to ensure the filtering mechanism works correctly.

In [57]:
# Import required libraries
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch


In [58]:
# Load GPT-2 model and tokenizer
model_name = "gpt2"  # You can also use "gpt2-medium"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [59]:
#  Define Python coding keywords for filtering
python_keywords = ["python", "code", "function", "class", "import", "variable", "loop", "def"]

#  Define the function to filter and generate responses
def answer_python_question(prompt, max_length=100):
    # Check if the prompt contains Python keywords
    if any(keyword.lower() in prompt.lower() for keyword in python_keywords):
        # Tokenize input
        inputs = tokenizer(prompt, return_tensors="pt").to(device)

        # Generate output
        outputs = model.generate(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            max_length=max_length,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.7,
            pad_token_id=tokenizer.eos_token_id
        )

        # Decode and return
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        return response
    else:
        return "Sorry, I can only answer questions related to Python coding."


In [60]:
#  Test examples
prompts = [
    "How to define a function in Python?",
    "What is the capital of France?",
    "Write a Python class for a bank account",
    "Tell me a joke",
    "How to use loops in Python?"
]

for prompt in prompts:
    print(f"Prompt: {prompt}")
    print(f"Response: {answer_python_question(prompt, max_length=50)}\n")


Prompt: How to define a function in Python?
Response: How to define a function in Python?

The Python language provides functions that are called with arguments and return values. A function definition can be defined with a list of arguments:

def __init__ ( self ): self . __init__

Prompt: What is the capital of France?
Response: Sorry, I can only answer questions related to Python coding.

Prompt: Write a Python class for a bank account
Response: Write a Python class for a bank account

For example, you could use a Python class for a bank account, which would be a Python class for a bank account and a Python class for a bank account.

class MyAccount ( object

Prompt: Tell me a joke
Response: Sorry, I can only answer questions related to Python coding.

Prompt: How to use loops in Python?
Response: How to use loops in Python?

I like to use loops in Python because they are easier to read and understand. I like to use looping to simplify the syntax of Python code.

What are some examp